<a href="https://colab.research.google.com/github/MaineCalabrezi13/DiagnosticoPlantas/blob/main/N%C3%A3o_folha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import shutil
import random

In [ ]:
# 1. Configurar a credencial do Kaggle (mantenha seu token ativo)
os.environ['KAGGLE_API_TOKEN'] = "KGAT_78d11ff3ddf72ec99f8d02cb4d33f2cc"

# 2. Baixar o novo dataset de folhas vs não folhas
!kaggle datasets download -d robiulhasanjisan/leaf-vs-non-leaf-images

# 3. Descompactar o arquivo no ambiente do Colab
!unzip -q leaf-vs-non-leaf-images.zip -d dataset_original
print("Prontinho! O novo dataset foi baixado e descompactado com sucesso.")

Dataset URL: https://www.kaggle.com/datasets/robiulhasanjisan/leaf-vs-non-leaf-images
License(s): MIT
leaf-vs-non-leaf-images.zip: Skipping, found more recently modified local copy (use --force to force download)
replace dataset_original/leaf/leaf_image0001.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
# 1. Caminhos de origem e destino
origem_dir = "dataset_original"
destino_base = "dataset_binario"

# Limpar estruturas antigas se existirem
if os.path.exists(destino_base):
    shutil.rmtree(destino_base)

# Criar a estrutura básica do YOLO
os.makedirs(os.path.join(destino_base, "train"), exist_ok=True)
os.makedirs(os.path.join(destino_base, "val"), exist_ok=True)

# 2. Encontrar as pastas reais de imagens varrendo tudo de forma flexível
pastas_encontradas = {}
for raiz, diretorios, arquivos in os.walk(origem_dir):
    # Se a pasta contiver qualquer arquivo (vamos listar tudo para garantir)
    if len(arquivos) > 0:
        nome_pasta = os.path.basename(raiz).lower()

        # Mapeia baseado em conter 'non' ou ser a pasta de folhas
        if "non" in nome_pasta or "not" in nome_pasta:
            pastas_encontradas["non-leaf"] = raiz
        elif "leaf" in nome_pasta:
            pastas_encontradas["leaf"] = raiz

print("Pastas mapeadas com sucesso:", pastas_encontradas)

# Se ainda assim vier vazio, pegamos as pastas do primeiro nível que existirem para não travar
if not pastas_encontradas:
    print("\nAviso: Mapeamento por nome falhou. Tentando capturar subpastas estruturais...")
    subpastas = [os.path.join(raiz, d) for raiz, dirs, _ in os.walk(origem_dir) for d in dirs]
    # Filtra apenas as pastas que estão no nível mais profundo e têm arquivos
    for sub in subpastas:
        arquivos = os.listdir(sub)
        if arquivos and os.path.isfile(os.path.join(sub, arquivos[0])):
            nome = os.path.basename(sub).lower()
            if "non" in nome or "not" in nome:
                pastas_encontradas["non-leaf"] = sub
            else:
                pastas_encontradas["leaf"] = sub
    print("Novo mapeamento forçado:", pastas_encontradas)

# Proporção de divisão (80% treino, 20% validação)
split_proporcao = 0.8

print("\nOrganizando as imagens em Folha vs Não Folha... Aguarde.")

# 3. Processar cada categoria encontrada
for cat, caminho_real in pastas_encontradas.items():
    os.makedirs(os.path.join(destino_base, "train", cat), exist_ok=True)
    os.makedirs(os.path.join(destino_base, "val", cat), exist_ok=True)

    # Lista TODOS os arquivos da pasta (reovendo arquivos ocultos de sistema como .DS_Store)
    imagens = [img for img in os.listdir(caminho_real) if not img.startswith('.')]
    random.shuffle(imagens)

    # Calcular o ponto de corte para 80/20
    limite = int(len(imagens) * split_proporcao)
    imagens_train = imagens[:limite]
    imagens_val = imagens[limite:]

    print(f" -> Categoria [{cat}]: {len(imagens_train)} arquivos para treino, {len(imagens_val)} para validação.")

    # Copiar os arquivos correspondentes para o treino
    for img in imagens_train:
        shutil.copy(os.path.join(caminho_real, img), os.path.join(destino_base, "train", cat, img))

    # Copiar os arquivos correspondentes para a validação
    for img in imagens_val:
        shutil.copy(os.path.join(caminho_real, img), os.path.join(destino_base, "val", cat, img))

print("\nSucesso! Dados prontos e organizados na pasta 'dataset_binario'.")

Pastas mapeadas com sucesso: {'leaf': 'dataset_original/leaf', 'non-leaf': 'dataset_original/non_leaf'}

Organizando as imagens em Folha vs Não Folha... Aguarde.
 -> Categoria [leaf]: 5200 arquivos para treino, 1300 para validação.
 -> Categoria [non-leaf]: 5200 arquivos para treino, 1300 para validação.

Sucesso! Dados prontos e organizados na pasta 'dataset_binario'.


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.9 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# 1. Carrega o modelo YOLO nano para classificação
model = YOLO('yolov8n-cls.pt')

# 2. Inicia o treinamento apontando para a nossa nova pasta base
results = model.train(
    data='dataset_binario', # Pasta que organizamos na Célula 3
    epochs=10,              # 10 épocas para problemas binários simples
    imgsz=320,              # Mantendo a resolução padrão leve usada anteriormente
    degrees=10,
    scale=0.2,
    fliplr=0.5,
    workers=2
)

print("TREINAMENTO DO CLASSIFICADOR BINÁRIO CONCLUÍDO COM SUCESSO!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_binario, degrees=10, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=